# Full Instance-Level TTA Gate (F1-1) — Colab runner

**Tujuan:** menjawab pertanyaan terbuka paper (Section V.F caveat) — apakah *gate* berbasis
**varians asli** antar-20-enumerasi (std / IQR) mengungguli *proxy* murah `|p_solo - p_TTA|`?
Sekaligus **menyimpan 20 prediksi mentah per molekul** sehingga caveat "per-variant predictions
were not retained" bisa dihapus dari paper.

**INFERENCE-ONLY** — memakai checkpoint ChemBERTa yang SUDAH dilatih. Tidak ada training ulang.

**Sumber data:** `hasil_outputs.zip` di Drive (`/content/drive/MyDrive/pharm_/hasil_outputs.zip`).
Zip ini dibuat dari isi folder `outputs/`, jadi begitu diekstrak ia berisi **checkpoint**
(`checkpoints/*.pt`) **dan prediksi tersimpan** (`predictions/*.npy`) sekaligus — sehingga skrip
jalan dalam mode *faithful* (angka solo/TTA **identik paper**).

**Langkah:** (1) Runtime → ubah tipe → **GPU**, (2) pastikan path zip benar di cell 4, (3) **Run All**.

In [ ]:
# 1. Cek GPU + mount Google Drive (tempat hasil_outputs.zip berada)
import torch, os
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'TIDAK ADA -> set Runtime>Change runtime type>GPU')
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone repo publik (KODE) ke /content/pharm_ + masuk foldernya
REPO = 'https://github.com/belvahector-ship-it/pharm_.git'
if not os.path.exists('/content/pharm_'):
    !git clone -q {REPO} /content/pharm_
%cd /content/pharm_
!git pull -q && git log --oneline -1

In [ ]:
# 3. Dependency — versi DIPIN ke environment run asli (torch bawaan Colab dipertahankan)
!pip install -q 'rdkit==2026.3.3' 'transformers==5.0.0' 'tokenizers==0.22.2' 'chemprop==2.2.4'
print('install selesai (versi dipin)')

In [ ]:
# 4. Ekstrak hasil_outputs.zip dari Drive -> tempatkan .pt & .npy ke folder repo yang benar.
#    >>> Jika path zip Anda beda, edit ZIP_PATH di bawah. <<<
ZIP_PATH = '/content/drive/MyDrive/pharm_/hasil_outputs.zip'
import zipfile, glob, shutil, os
assert os.path.exists(ZIP_PATH), f'Zip tak ditemukan: {ZIP_PATH} — cek path di Drive Anda.'
EXTRACT_DIR = '/content/_extracted'
if not os.path.exists(EXTRACT_DIR):
    print(f'Mengekstrak {ZIP_PATH} ...')
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)
    print('ekstraksi selesai.')
os.makedirs('outputs/checkpoints', exist_ok=True)
os.makedirs('outputs/predictions', exist_ok=True)
# Struktur internal zip bisa 'checkpoints/..' atau 'outputs/checkpoints/..' -> cari rekursif.
n_ck = n_pr = 0
for pt in glob.glob(os.path.join(EXTRACT_DIR, '**', '*.pt'), recursive=True):
    d = os.path.join('outputs/checkpoints', os.path.basename(pt))
    if not os.path.exists(d): shutil.copy(pt, d); n_ck += 1
for npy in glob.glob(os.path.join(EXTRACT_DIR, '**', '*.npy'), recursive=True):
    d = os.path.join('outputs/predictions', os.path.basename(npy))
    if not os.path.exists(d): shutil.copy(npy, d); n_pr += 1
ck = sorted(os.path.basename(x) for x in glob.glob('outputs/checkpoints/chemberta_*.pt'))
print(f'{n_ck} checkpoint (.pt) & {n_pr} prediksi (.npy) ditempatkan.')
print(f'Checkpoint ChemBERTa siap ({len(ck)}):'); print('\n'.join(ck) if ck else '(TIDAK ADA — cek isi zip!)')
assert ck, 'Tidak ada checkpoint chemberta_*.pt di dalam zip — periksa isi hasil_outputs.zip.'

In [ ]:
# 5. Jalankan eksperimen F1-1 (semua dataset, 10 seed, backbone base). ~10-20 menit di T4.
#    --save_raw menyimpan 20 varian mentah (menghapus caveat 'not retained').
#    Hanya kombinasi yang checkpoint-nya ada yang diproses; sisanya di-skip otomatis.
!python scripts/14_full_instance_gate.py --backbones chemberta --save_raw 2>&1 | grep -vE 'Explicit valence|Loading weights|HF_TOKEN|RuntimeWarning|Degrees of freedom'

In [ ]:
# 6. Lihat hasil ringkas: apakah std / IQR (varians asli) mengalahkan proxy disagreement?
import pandas as pd
s = pd.read_csv('outputs/results/full_gate/full_instance_gate_summary.csv')
pd.set_option('display.width', 200, 'display.max_columns', 50)
print(s.to_string(index=False))
print('\n--- interpretasi ---')
print('Bandingkan kolom gate_disagree vs gate_std vs gate_iqr per dataset.')
print('Jika proxy (disagree) >= std & iqr -> temuan paper KONFIRMASI: sinyal murah cukup.')
print('Jika std/iqr > disagree signifikan -> varians asli menang -> update Table V + klaim.')

In [ ]:
# 7. Simpan hasil kembali ke Drive (CSV + report + varian mentah)
OUT_DRIVE = '/content/drive/MyDrive/pharm_full_gate_results'
os.makedirs(OUT_DRIVE, exist_ok=True)
for f in glob.glob('outputs/results/full_gate/*'):
    shutil.copy(f, OUT_DRIVE)
raws = glob.glob('outputs/predictions/chemberta_tta_raw_*.npz')
os.makedirs(os.path.join(OUT_DRIVE, 'raw_variants'), exist_ok=True)
for f in raws:
    shutil.copy(f, os.path.join(OUT_DRIVE, 'raw_variants'))
print(f'Hasil disalin ke {OUT_DRIVE} (+{len(raws)} berkas varian mentah).')
print('Kirim balik full_instance_gate_summary.csv untuk saya masukkan ke paper bila perlu.')